# GR00T humanoid robot, served by Ray Serve on Anyscale

NVIDIA's GR00T-N1.7-3B vision-language-action model controlling a Unitree G1 humanoid in NVIDIA Isaac Lab simulation, served at scale via Ray Serve on Anyscale.

## Overview

Three things happen end to end:

1. The GR00T-N1.7-3B foundation model is deployed behind an HTTP endpoint with Ray Serve
2. The policy responds to observations with correctly-shaped action chunks at sub-second latency
3. A pre-recorded rollout from Isaac Lab is displayed inline

## Why this matters

Anyscale is the platform for running modern AI workloads at scale. Robotics foundation models like GR00T combine three hard things at once:

- **Heavy GPU inference** for a 3B parameter VLA (vision-language-action) model
- **Realtime physics simulation** with NVIDIA Isaac Lab and Isaac Sim
- **Multi-machine orchestration** with the policy and simulator on separate GPUs, communicating over HTTP

Ray Serve handles GR00T inference scaling. Ray tasks fan out the Isaac Lab simulators. Anyscale runs the cluster. The same primitives that scale LLM inference scale cleanly to robotics.

## Architecture

<div align="center">

<svg xmlns="http://www.w3.org/2000/svg" width="900" viewBox="0 0 720 460" style="max-width: 100%; height: auto;">
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#666" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Cluster boundary -->
  <rect x="20" y="40" width="680" height="260" rx="14" fill="none" stroke="#888780" stroke-width="0.5" stroke-dasharray="4 4"/>
  <text x="40" y="64" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#2C2C2A">Anyscale Ray cluster</text>

  <!-- Head node -->
  <rect x="50" y="160" width="170" height="120" rx="10" fill="#FAEEDA" stroke="#854F0B" stroke-width="0.5"/>
  <text x="135" y="188" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#633806">Head node</text>
  <text x="135" y="208" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#854F0B">Jupyter notebook</text>
  <text x="135" y="226" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#854F0B">Ray driver</text>
  <text x="135" y="244" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#854F0B">Schedules tasks</text>
  <text x="135" y="262" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#854F0B">Renders the GIF</text>

  <!-- GPU worker A (policy) -->
  <rect x="270" y="150" width="200" height="160" rx="10" fill="#EEEDFE" stroke="#534AB7" stroke-width="0.5"/>
  <text x="370" y="178" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#3C3489">GPU worker A</text>
  <text x="370" y="198" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#534AB7">running Ray Serve</text>
  <rect x="290" y="212" width="160" height="84" rx="6" fill="#CECBF6" stroke="#534AB7" stroke-width="1" fill-opacity="0.5"/>
  <text x="370" y="232" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#3C3489">GR00T-N1.7-3B</text>
  <text x="370" y="250" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#534AB7">VLA policy on cuda:0</text>
  <text x="370" y="268" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#534AB7">FastAPI ingress</text>
  <text x="370" y="286" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#534AB7">POST /predict</text>

  <!-- GPU worker B (sim) -->
  <rect x="500" y="150" width="180" height="160" rx="10" fill="#E1F5EE" stroke="#0F6E56" stroke-width="0.5"/>
  <text x="590" y="178" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#085041">GPU worker B</text>
  <text x="590" y="198" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#0F6E56">running Ray task</text>
  <rect x="518" y="212" width="144" height="84" rx="6" fill="#9FE1CB" stroke="#0F6E56" stroke-width="1" fill-opacity="0.5"/>
  <text x="590" y="232" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#085041">Isaac Lab</text>
  <text x="590" y="250" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#0F6E56">Isaac Sim 5.1</text>
  <text x="590" y="268" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#0F6E56">Unitree G1 humanoid</text>
  <text x="590" y="286" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#0F6E56">pick-place scene</text>

  <!-- deploy arrow -->
  <line x1="220" y1="180" x2="266" y2="180" stroke="#666" stroke-width="1.2" fill="none" marker-end="url(#arrow)"/>
  <text x="243" y="173" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">deploy</text>

  <!-- launch sim task arrow -->
  <path d="M 135 156 L 135 110 L 590 110 L 590 146" stroke="#666" stroke-width="1.2" fill="none" marker-end="url(#arrow)"/>
  <text x="362" y="103" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">launch sim task</text>

  <!-- obs / actions -->
  <line x1="496" y1="188" x2="474" y2="188" stroke="#666" stroke-width="1.2" fill="none" marker-end="url(#arrow)"/>
  <text x="485" y="181" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">obs</text>

  <line x1="474" y1="270" x2="496" y2="270" stroke="#666" stroke-width="1.2" fill="none" marker-end="url(#arrow)"/>
  <text x="485" y="264" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">actions</text>

  <text x="485" y="227" text-anchor="middle" font-family="-apple-system, sans-serif" font-size="12" fill="#888780">HTTP</text>

  <!-- Legend -->
  <rect x="20" y="320" width="680" height="120" rx="8" fill="none" stroke="#D3D1C7" stroke-width="0.5"/>
  <text x="36" y="342" font-family="-apple-system, sans-serif" font-size="14" font-weight="500" fill="#2C2C2A">Legend</text>

  <rect x="36" y="356" width="14" height="14" rx="3" fill="#FAEEDA" stroke="#854F0B" stroke-width="0.5"/>
  <text x="58" y="367" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">Head node hosting the notebook</text>

  <rect x="36" y="380" width="14" height="14" rx="3" fill="#EEEDFE" stroke="#534AB7" stroke-width="0.5"/>
  <text x="58" y="391" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">Ray Serve replica hosting GR00T</text>

  <rect x="36" y="404" width="14" height="14" rx="3" fill="#E1F5EE" stroke="#0F6E56" stroke-width="0.5"/>
  <text x="58" y="415" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">Isaac Lab simulator with Unitree G1</text>

  <rect x="380" y="356" width="14" height="14" rx="3" fill="none" stroke="#888780" stroke-dasharray="3 3" stroke-width="0.5"/>
  <text x="402" y="367" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">Anyscale cluster boundary</text>

  <line x1="380" y1="389" x2="394" y2="389" stroke="#666" stroke-width="1.2" fill="none" marker-end="url(#arrow)"/>
  <text x="402" y="393" font-family="-apple-system, sans-serif" font-size="12" fill="#5F5E5A">control / data flow</text>
</svg>

</div>

## Step 0: Hugging Face authentication

GR00T-N1.7 uses `nvidia/Cosmos-Reason2-2B` as its vision-language backbone. The Cosmos model is gated on Hugging Face, so a token with access is required. Terms must be accepted at https://huggingface.co/nvidia/Cosmos-Reason2-2B.

In [ ]:
import os

os.environ["HF_TOKEN"] = "REDACTED"
HF_TOKEN = os.environ["HF_TOKEN"]
assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "hf_...", (
    "Set HF_TOKEN to a real token. Get one at https://huggingface.co/settings/tokens"
)
print(f"HF token loaded (ends in ...{HF_TOKEN[-4:]})")

## Step 1: Connect to the Ray cluster

Attach to the running Anyscale cluster. Ray's `runtime_env` carries the HF token to every Ray task.

In [ ]:
import ray

ray.init(
    address="auto",
    ignore_reinit_error=True,
    runtime_env={"env_vars": {"HF_TOKEN": HF_TOKEN}},
)

resources = ray.cluster_resources()
print(f"GPUs:   {int(resources.get('GPU', 0))}")
print(f"CPUs:   {int(resources.get('CPU', 0))}")
print(f"Memory: {resources.get('memory', 0) / 1e9:.0f} GB")

## Step 2: Pre-warm the model cache

GR00T-N1.7 loads two checkpoints at deploy time:

- `nvidia/GR00T-N1.7-3B`, the policy itself (~6 GB)
- `nvidia/Cosmos-Reason2-2B`, the gated VLM backbone (~5 GB)

Pre-downloading both to every GPU worker means Ray Serve can land the replica on any worker without a cold-start surprise. With models pre-cached in the cluster image, this completes in seconds.

In [ ]:
@ray.remote(num_gpus=1, runtime_env={"env_vars": {"HF_TOKEN": HF_TOKEN}})
def prewarm_models():
    import os, time
    from huggingface_hub import snapshot_download
    t0 = time.time()
    snapshot_download("nvidia/GR00T-N1.7-3B")
    snapshot_download("nvidia/Cosmos-Reason2-2B")
    return f"{os.uname().nodename}: ready in {time.time()-t0:.0f}s"

n_gpus = int(ray.cluster_resources().get("GPU", 0))
print(f"Pre-warming GR00T and Cosmos on {n_gpus} GPU workers in parallel...")
print()
for r in ray.get([prewarm_models.remote() for _ in range(n_gpus)]):
    print(f"  {r}")

## Step 3: Deploy GR00T behind an HTTP endpoint

`GR00TPolicyServer` is a Python class wrapped in `@serve.deployment` and `@serve.ingress(FastAPI())`. With one `serve.run` call, Ray Serve:

- Schedules a replica on a GPU worker
- Loads the 3B parameter GR00T-N1.7-3B model onto cuda:0
- Stands up an HTTP server with `POST /predict` and `GET /stats`
- Returns a stable URL the rest of the cluster can reach

Loading a 3B parameter model to GPU memory takes longer than Ray Serve's default 30 second health-check timeout, so `health_check_timeout_s` is extended. `HF_TOKEN` is passed through `ray_actor_options` so the replica's process inherits it.

Expected runtime with pre-cached weights: 60 to 90 seconds.

In [ ]:
import sys
import time
from ray import serve

sys.path.insert(0, "path_a_ray_serve")
from policy_server import GR00TPolicyServer

print("Deploying GR00TPolicyServer to Ray Serve...")

deployment = GR00TPolicyServer.options(
    num_replicas=1,
    health_check_timeout_s=300,
    health_check_period_s=120,
    graceful_shutdown_timeout_s=10,
    ray_actor_options={
        "num_gpus": 1,
        "runtime_env": {"env_vars": {"HF_TOKEN": HF_TOKEN}},
    },
).bind(
    model_path="nvidia/GR00T-N1.7-3B",
    embodiment_tag="REAL_G1",
)

serve.start(detached=False, http_options={"host": "0.0.0.0", "port": 8000})
serve.run(deployment, name="gr00t-policy")

import socket
def _head_ip():
    try:
        return ray.get_runtime_context().gcs_address.split(":")[0]
    except Exception:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect(("8.8.8.8", 80))
            return s.getsockname()[0]

POLICY_URL = f"http://{_head_ip()}:8000"
print()
print(f"Policy is live at {POLICY_URL}")

## Step 4: Send the policy a real observation

A real observation in GR00T's REAL_G1 schema includes:

- A short stack of camera frames (`video.ego_view`)
- The robot's joint state across both arms, hands, and waist
- A natural language instruction

The policy returns a 40-step action chunk covering arms, hands, waist, base height, and navigation commands. This is the round-trip Isaac Lab makes once per chunk during a rollout.

In [ ]:
import numpy as np
import pickle
import requests

identity_pose = np.array([0.3, 0.0, 0.0, 1, 0, 0, 0, 1, 0], dtype=np.float32)
dummy_obs = {
    "video": {"ego_view": np.zeros((1, 2, 256, 256, 3), dtype=np.uint8)},
    "state": {
        "left_wrist_eef_9d":  identity_pose[None, None, :].copy(),
        "right_wrist_eef_9d": identity_pose[None, None, :].copy(),
        "left_arm":   np.zeros((1, 1, 7), dtype=np.float32),
        "right_arm":  np.zeros((1, 1, 7), dtype=np.float32),
        "left_hand":  np.zeros((1, 1, 7), dtype=np.float32),
        "right_hand": np.zeros((1, 1, 7), dtype=np.float32),
        "waist":      np.zeros((1, 1, 3), dtype=np.float32),
    },
    "language": {
        "annotation.human.task_description": [["pick up the apple and place it on the plate"]]
    },
}

t0 = time.time()
r = requests.post(f"{POLICY_URL}/predict", data=pickle.dumps(dummy_obs), timeout=180)
r.raise_for_status()
resp = pickle.loads(r.content)
latency_ms = (time.time() - t0) * 1000

print(f"Round trip: {latency_ms:.0f} ms")
print()
print("Action chunk:")
for k, v in resp["action"].items():
    print(f"  {k:24s} {np.asarray(v).shape}")

## Step 5: A full simulation rollout

The end-to-end flow stitches the policy and Isaac Lab's Unitree G1 pick-place scene together:

1. Isaac Lab boots and loads the G1 with table and apple
2. Each step, the simulator captures camera frames and joint state
3. Every few steps, the observation is sent to the same Ray Serve endpoint deployed above
4. The 40-step action chunk returns, gets translated into the simulator's `(1, 28)` action space, and steps the physics
5. Camera frames stack into a GIF

The clip below is a pre-recorded rollout. The same code runs on a separate Ray task in production, scaling to many parallel rollouts.

In [ ]:
from IPython.display import Image, display
import os

gif_path = "g1_groot_n17_zeroshot.gif"

if os.path.exists(gif_path):
    print(f"Displaying {gif_path}")
    print()
    display(Image(filename=gif_path))
else:
    print(f"GIF not found at {gif_path}")
    print("Update gif_path to the location of the demo GIF in this repo.")

### Reading the rollout

Three things are worth noticing in the clip above:

- The robot is in the right scene with the right embodiment. Isaac Lab loaded the Unitree G1 and placed it at the pick-place table with a target apple, rendering camera frames at the resolution GR00T expects.
- Every motion is the policy's own output. There is no scripted joint trajectory and no replay. The arms move because GR00T returned a 40-step action chunk and the simulator stepped through it.
- The motion is exploratory rather than task-completing. The arms search the workspace, but do not yet land a clean grasp on the apple.

The clip is a **zero-shot rollout from the GR00T-N1.7-3B base model**. Two simple changes typically improve the rollout substantially:

1. **Wire real joint state into the observation.** This rollout feeds zeroed joint positions. The policy was trained to read its own arm and hand state. With Isaac Lab's real `robot.data.joint_pos` mapped to the GR00T schema, the actions become much more directed.
2. **Use the published G1 fine-tune.** NVIDIA released `nvidia/GR00T-N1.6-G1-PnPAppleToPlate`, a checkpoint specifically post-trained on this exact pick-place task. That is Path B in this repo and is the path to a clean grasp.

Both improvements land naturally on the same Ray Serve infrastructure shown in Step 3. The model swap is a one-line change. The platform stays the same.

## Step 6: Inspect the policy server

Ray Serve exposes the policy as a real HTTP service. The stats endpoint reports latency and call count over the deployment's lifetime.

In [ ]:
import json
r = requests.get(f"{POLICY_URL}/stats", timeout=10)
print(json.dumps(r.json(), indent=2))

## Going further

### Scale the policy server horizontally

```python
deployment = GR00TPolicyServer.options(num_replicas=4).bind(...)
```

Ray Serve schedules each replica on its own GPU. Sim workers load-balance across them automatically.

### Run many sim rollouts in parallel

```python
results = ray.get([run_sim_rollout.remote(POLICY_URL) for _ in range(100)])
```

Each rollout grabs a GPU worker, queries the shared policy fleet, and saves its own GIF.

### The G1 fine-tune (Path B)

This notebook used GR00T-N1.7-3B base. The repo also includes Path B, which loads NVIDIA's published G1 pick-place fine-tune `nvidia/GR00T-N1.6-G1-PnPAppleToPlate` in a separate conda env, file-bridged to Isaac Lab.

```bash
bash path_b_file_bridge/orchestrate_n16.sh
```

## Cleanup

Tear down the Ray Serve deployment. The Ray cluster keeps running.

In [ ]:
serve.shutdown()
print("Ray Serve stopped.")